# 01 · Memória de longo prazo: o básico

O que este notebook mostra:

1. criar um `PiMemory` com os motores locais (FastEmbed + Ollama + Qdrant numa pasta);
2. entregar conversas e ver **cada decisão** (guardar, substituir, apagar, ignorar);
3. buscar, listar, editar e apagar memórias à mão;
4. a separação por **dono** (`owner`).

> **Instalação** (uma vez):
> ```bash
> pip install "pi-memory[local] @ git+https://github.com/Pedro-Bianco/pi-memory"
> ollama pull qwen2.5:14b
> ```
> Todos os dados deste notebook são fictícios. As saídas abaixo vêm de uma execução real.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm no Jupyter

def mostrar(obj, limite=1500):
    """JSON legível (o mesmo formato da biblioteca, do HTTP e do MCP)."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    print(texto if len(texto) <= limite else texto[:limite] + "\n…")

def pasta_limpa(nome):
    """Cada notebook começa do zero: apaga a pasta de dados da execução anterior."""
    shutil.rmtree(nome, ignore_errors=True)
    return nome

## Criar o objeto

Tudo parte de um objeto que você cria: não há estado global. `path` é a pasta do banco local
(Qdrant embutido). O modelo de linguagem padrão já é o `qwen2.5:14b`, o que passou na avaliação
registrada (veja o README); aqui ele é passado explicitamente só para ficar visível.

In [2]:
from pi_memory import PiMemory
from pi_memory.llms import OllamaLLM

pm = PiMemory(path=pasta_limpa("dados_01"), llm=OllamaLLM("qwen2.5:14b", timeout=180))
print(pm.embedder.name, "|", pm.embedder.dimension, "dimensões")

fastembed:intfloat/multilingual-e5-large | 1024 dimensões


## Guardar a partir de uma conversa

`memories.add` recebe uma conversa (lista de mensagens `user`/`assistant`, ou um texto solto
como mensagem do usuário). O modelo extrai os fatos sobre o usuário e, para cada um, decide o
que fazer comparando com as 3 memórias mais parecidas do mesmo dono. O retorno lista as
decisões.

In [3]:
r = pm.memories.add("Oi! Meu nome é Ana Exemplo, moro em Curitiba e trabalho como enfermeira.",
                    owner="ana")
mostrar(r)

{
  "decisions": [
    {
      "event": "add",
      "memory_id": "e9577129-aa25-4cef-87b9-127843ed6c04",
      "text": "O nome do usuário é Ana Exemplo",
      "previous_text": null,
      "topic": "perfil",
      "reason": null
    },
    {
      "event": "add",
      "memory_id": "bf9cbd38-d413-4cc6-b1b1-2e0338595ae5",
      "text": "O usuário mora em Curitiba",
      "previous_text": null,
      "topic": "casa",
      "reason": null
    },
    {
      "event": "add",
      "memory_id": "2acc9f8f-ec3d-47e1-bb55-01971c4b1bf2",
      "text": "O usuário trabalha como enfermeira",
      "previous_text": null,
      "topic": "trabalho",
      "reason": null
    }
  ]
}


## Substituir: um fato novo sobre o mesmo assunto

"Me mudei" deve **substituir** a cidade antiga (evento `update`, com `previous_text`), não criar
uma segunda memória.

In [4]:
mostrar(pm.memories.add("Novidade: me mudei para Recife semana passada.", owner="ana"))

{
  "decisions": [
    {
      "event": "update",
      "memory_id": "bf9cbd38-d413-4cc6-b1b1-2e0338595ae5",
      "text": "O usuário se mudou para Recife semana passada.",
      "previous_text": "O usuário mora em Curitiba",
      "topic": "casa",
      "reason": null
    }
  ]
}


## Apagar: algo deixou de ser verdade

Pedir para esquecer, ou contar que algo acabou, gera `delete`. Na dúvida a biblioteca **guarda**
e nunca apaga por palpite (o campo `reason` aparece quando isso acontece).

In [5]:
pm.memories.remember("O usuário tem um carro", owner="ana")      # fato direto, sem modelo
mostrar(pm.memories.add("Vendi meu carro, agora só ando de bicicleta.", owner="ana"))

{
  "decisions": [
    {
      "event": "delete",
      "memory_id": "70f33634-40e8-43d8-a2b6-c0366f03440a",
      "text": "O usuário vendeu o carro e não tem mais carro",
      "previous_text": "O usuário tem um carro",
      "topic": "outros",
      "reason": null
    },
    {
      "event": "add",
      "memory_id": "4c54675f-4ed9-459a-9935-69a60c534ffe",
      "text": "O usuário agora só anda de bicicleta",
      "previous_text": null,
      "topic": "rotina",
      "reason": null
    }
  ]
}


## Ignorar: nada novo, ou nada sobre o usuário

In [6]:
mostrar(pm.memories.add("Como eu disse, sou enfermeira.", owner="ana"))
mostrar(pm.memories.add("Qual a capital da Austrália?", owner="ana"))

{
  "decisions": [
    {
      "event": "none",
      "memory_id": null,
      "text": "O usuário é enfermeira.",
      "previous_text": null,
      "topic": null,
      "reason": null
    }
  ]
}


{
  "decisions": []
}


## Conversa com o assistente como contexto

As falas do assistente servem só de contexto (para entender um "sim"); nada que **só** o
assistente disse vira memória.

In [7]:
mostrar(pm.memories.add([
    {"role": "assistant", "content": "Você ainda faz plantão noturno?"},
    {"role": "user", "content": "sim, às quartas e sextas"},
], owner="ana"))

{
  "decisions": [
    {
      "event": "add",
      "memory_id": "906a11d4-f528-4dc2-b49e-3e525b21b3a3",
      "text": "O usuário faz plantão noturno às quartas e sextas",
      "previous_text": null,
      "topic": "trabalho",
      "reason": null
    }
  ]
}


## Buscar

Busca híbrida: significado + palavra exata. Devolve as memórias mais próximas com pontuação;
use as 3 primeiras como contexto, não só a primeira.

In [8]:
for r in pm.memories.search("onde a Ana mora?", owner="ana")["results"]:
    print(f"{r['score']:.3f}  [{r['memory']['topic']}]  {r['text']}")

1.289  [perfil]  O nome do usuário é Ana Exemplo
0.479  [casa]  O usuário se mudou para Recife semana passada.
0.449  [trabalho]  O usuário trabalha como enfermeira


## Listar, editar e apagar à mão

In [9]:
lista = pm.memories.list(owner="ana")
print("total:", lista["total"])
for m in lista["memories"]:
    print(f"- [{m['topic']}] {m['text']}")

total: 5
- [trabalho] O usuário faz plantão noturno às quartas e sextas
- [rotina] O usuário agora só anda de bicicleta
- [casa] O usuário se mudou para Recife semana passada.
- [trabalho] O usuário trabalha como enfermeira
- [perfil] O nome do usuário é Ana Exemplo


In [10]:
primeira = lista["memories"][0]
editada = pm.memories.update(primeira["id"], owner="ana", text=primeira["text"] + " (confirmado)")
print(editada["text"])
print(pm.memories.delete(primeira["id"], owner="ana"))

O usuário faz plantão noturno às quartas e sextas (confirmado)
{'deleted': 1}


## Donos separados

Cada operação recebe `owner`. As memórias de um dono não aparecem para outro, e um `id` de
outro dono é tratado como inexistente.

In [11]:
pm.memories.remember("O usuário joga xadrez", owner="bruno")
print("ana:", [r["text"] for r in pm.memories.search("xadrez", owner="ana")["results"][:1]])
print("bruno:", [r["text"] for r in pm.memories.search("xadrez", owner="bruno")["results"][:1]])

from pi_memory import NotFoundError
try:
    pm.memories.get(lista["memories"][1]["id"], owner="bruno")
except NotFoundError as e:
    print("NotFoundError:", e.message)

ana: ['O nome do usuário é Ana Exemplo']
bruno: ['O usuário joga xadrez']
NotFoundError: Memória '4c54675f-4ed9-459a-9935-69a60c534ffe' não encontrada para o dono 'bruno'.


## Erros claros, em português

Toda exceção da biblioteca tem `code` e `message`; `to_dict()` dá o mesmo JSON de erro que o
HTTP e o MCP vão devolver.

In [12]:
from pi_memory import PiMemoryError
try:
    pm.memories.add("oi", owner="")
except PiMemoryError as e:
    mostrar(e.to_dict())

{
  "error": {
    "code": "invalid_input",
    "message": "Informe o dono (owner): um texto não vazio, ex. 'ana'."
  }
}


In [13]:
pm.close()